# Code 8.5.4: Where the pure sample left the nucleus

Finds the rank and probability of the token at which the pure sample of Code 8.5.2 diverged from the top-p sample, and the size of the top-p set ($p = 0.95$) at that step.


In [ ]:
# Where the pure sample of Code 8.5.2 left the top-p set: rank and probability of its token
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
tok = GPT2TokenizerFast.from_pretrained("gpt2"); model = GPT2LMHeadModel.from_pretrained("gpt2").eval()
prompt = "The city council met on Tuesday night to discuss the plan for the new park."
cont = ' The existing seven-story building has been "tree-lined" on both sides, and new condos would include retail and restaurants.\n\nThe change would lower parking in a large downtown'
ids = tok(prompt + cont, return_tensors="pt").input_ids
with torch.no_grad(): p = model(ids).logits[0, -1].softmax(-1)
probs, order = p.sort(descending=True)
nuc = int(((probs.cumsum(-1) - probs) < 0.95).sum())
for w in [" spike", " area"]:
    t = tok.encode(w)[0]; rank = int((order == t).nonzero())
    print(repr(w), f"p={p[t]:.2e} rank={rank + 1} nucleus size={nuc}")
# ' spike' p=4.40e-06 rank=2444 nucleus size=317
# ' area' p=2.20e-01 rank=1 nucleus size=317
